In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/gdrive')

# Change working directory to your file position
folder = '/content/gdrive/My Drive/WWF_2026/'
os.chdir(folder)

# Confirm the change
print("Working Directory:", os.getcwd())

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Working Directory: /content/gdrive/My Drive/WWF_2026


In [ ]:
# Subfolder where the .docx files are located
docx_subfolder = 'data/negatives' # Change this accordingly
docx_folder_path = os.path.join(os.getcwd(), docx_subfolder)

print(f"Checking for .docx files in: {docx_folder_path}")


docx_files = [f for f in os.listdir(docx_folder_path) if f.endswith('.DOCX') and not f.endswith('_doclist.DOCX')]

if not docx_files:
    print("No .docx files found in the specified folder.")
else:
    # Create a lookup dictionary with unique IDs
    docx_lookup = {}
    for i, filename in enumerate(docx_files):
        unique_id = f"nDOC_{i+1:03d}" # e.g., DOC_001, DOC_002
        docx_lookup[unique_id] = os.path.join(docx_folder_path, filename)

    print("\n--- DOCX File Lookup Dictionary ---")
    for doc_id, doc_path in docx_lookup.items():
        print(f"{doc_id}: {doc_path}")

    print("\nNumber of .DOCX files found and indexed:", len(docx_lookup))


Checking for .docx files in: /content/gdrive/My Drive/WWF_2026/data/negatives

--- DOCX File Lookup Dictionary ---
nDOC_001: /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzania ; Tanzanian Beaches Are Treasure Troves.DOCX
nDOC_002: /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzania ; Tanzania Plans to Lift Export Ban On Sea Fish.DOCX
nDOC_003: /content/gdrive/My Drive/WWF_2026/data/negatives/Health Danger in Dar Over Dynamite Fishing.DOCX
nDOC_004: /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzanian fishermen embark on efforts to conserve sea cows.DOCX
nDOC_005: /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzania Health Danger in Dar Over Dynamite Fishing.DOCX
nDOC_006: /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzania Dynamite Fishing Reported Thriving in Tanga Region.DOCX
nDOC_007: /content/gdrive/My Drive/WWF_2026/data/negatives/Spots Notorious for Illegal Fishing Exposed in Temeke.DOCX
nDOC_008: /content/gdrive/My Drive/WWF_2026/data/negatives/Tanz

In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.0 MB/s eta 0:00:00


In [ ]:
from docx import Document

def extract_text_from_docx(docx_path):
    """Extracts all text from a .docx file."""
    try:
        document = Document(docx_path)
        full_text = []
        for paragraph in document.paragraphs:
            full_text.append(paragraph.text)
        return '\n'.join(full_text)
    except Exception as e:
        return f"Error extracting text: {e}"


In [ ]:
### Text extraction test ###
# docx_file_name = '/content/gdrive/MyDrive/WWF_2026/data/raw/Tanzania (12_07).DOCX'
# extracted_content = extract_text_from_docx(docx_file_name)
# print(extracted_content)

### Structuring Extracted Content

In [ ]:
import datetime
import re

def parse_document_for_structured_json(doc_id, article_text):
    """
    Parses the article text to extract title, source, and load_date,
    and returns a structured dictionary including doc_id and article_text.
    """
    title = "Untitled Document"
    source = "Unknown Source"
    load_date = datetime.date.today().isoformat() # Default to current date

    # Clean up the text for easier parsing
    cleaned_text_lines = [line.strip() for line in article_text.split('\n') if line.strip()]

    if not cleaned_text_lines:
        return {
            "doc_id": doc_id,
            "title": title,
            "source": source,
            "load_date": load_date,
            "article_text": article_text
        }

    # Attempt to extract title and source from the first few non-empty lines.
    # This is a heuristic that assumes the first non-empty line is the title
    # and the second non-empty line is the source.
    if len(cleaned_text_lines) > 0:
        title = cleaned_text_lines[0]

    if len(cleaned_text_lines) > 1:
        source = cleaned_text_lines[1]

    # Extract load_date from the article_text if available
    load_date_match = re.search(r"Load-Date:\s*([A-Za-z]+ \d{1,2}, \d{4})", article_text)
    if load_date_match:
        # Convert the extracted date string to ISO format
        date_str = load_date_match.group(1)
        try:
            # Try to parse with full month name
            parsed_date = datetime.datetime.strptime(date_str, '%B %d, %Y').date()
        except ValueError:
            try:
                # If not, try with abbreviated month name
                parsed_date = datetime.datetime.strptime(date_str, '%b %d, %Y').date()
            except ValueError:
                # If both fail, keep default
                pass
        else:
            load_date = parsed_date.isoformat()


    return {
        "doc_id": doc_id,
        "title": title,
        "source": source,
        "load_date": load_date,
        "article_text": article_text
    }

In [ ]:
### Extracted json test ###
# json_out = parse_document_for_structured_json("DOC_020", extracted_content)
# print(json_out)

In [ ]:
# Define the output directory
output_dir = os.path.join(os.getcwd(), 'data/n_extracted/')

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured directory exists: {output_dir}")

Ensured directory exists: /content/gdrive/MyDrive/WWF_2026/data/n_extracted


In [ ]:
import json

# Iterate through the lookup dictionary and extract text
for doc_id, doc_path in docx_lookup.items():
  print(f"Extracting text from {doc_path} (ID: {doc_id})...")
  content = extract_text_from_docx(doc_path)
  extracted_json = parse_document_for_structured_json(doc_id, content)

  # Output JSON file path
  output_json_path = os.path.join(output_dir, f'{doc_id}_ext.json')

  with open(output_json_path, 'w', encoding='utf-8') as f:
      json.dump(extracted_json, f, ensure_ascii=False, indent=4)

  print(f"Extracted content saved to {output_json_path}\n")


Extracting text from /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzania ; Tanzanian Beaches Are Treasure Troves.DOCX (ID: nDOC_001)...
Extracted content saved to /content/gdrive/MyDrive/WWF_2026/data/n_extracted/nDOC_001_ext.json

Extracting text from /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzania ; Tanzania Plans to Lift Export Ban On Sea Fish.DOCX (ID: nDOC_002)...
Extracted content saved to /content/gdrive/MyDrive/WWF_2026/data/n_extracted/nDOC_002_ext.json

Extracting text from /content/gdrive/My Drive/WWF_2026/data/negatives/Health Danger in Dar Over Dynamite Fishing.DOCX (ID: nDOC_003)...
Extracted content saved to /content/gdrive/MyDrive/WWF_2026/data/n_extracted/nDOC_003_ext.json

Extracting text from /content/gdrive/My Drive/WWF_2026/data/negatives/Tanzanian fishermen embark on efforts to conserve sea cows.DOCX (ID: nDOC_004)...
Extracted content saved to /content/gdrive/MyDrive/WWF_2026/data/n_extracted/nDOC_004_ext.json

Extracting text from /content/gdrive

## Aggregate into 1 file if necessary

In [ ]:
import glob
import json
import os

# Ensure output_dir is defined (it should be from previous cells)
# If running this cell independently, uncomment and define it:
# output_dir = os.path.join(os.getcwd(), 'data', 'n_extracted')

all_extracted_data = []

# Get a list of all JSON files in the output directory
json_files = glob.glob(os.path.join(output_dir, '*_ext.json'))

print(f"Aggregating {len(json_files)} JSON files from: {output_dir}")

for json_file in json_files:
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            all_extracted_data.append(data)
    except Exception as e:
        print(f"Error reading {json_file}: {e}")

# Define the path for the aggregated metadata file
metadata_output_path = os.path.join(output_dir, 'metadata.json')

# Save all aggregated data to a single JSON file
with open(metadata_output_path, 'w', encoding='utf-8') as f:
    json.dump(all_extracted_data, f, ensure_ascii=False, indent=4)

print(f"All extracted JSON contents aggregated into: {metadata_output_path}")
print(f"Total documents aggregated: {len(all_extracted_data)}")

Aggregating 20 JSON files from: /content/gdrive/MyDrive/WWF_2026/data/n_extracted
All extracted JSON contents aggregated into: /content/gdrive/MyDrive/WWF_2026/data/n_extracted/metadata.json
Total documents aggregated: 20
